In [ ]:
import math
import numpy as np

## GLOBAL VARIABLES

# 1. Load Parameters
F_PROBE_LBF = 2.5            # Estimated downward force (weight) of the probe in lbf

# 2. Geometric Parameters
D_CANTILEVER_MM = 40.0       # Distance from probe center of mass to front bearing (Configuration 1)
L_BEARING_SPACE_MM = 15.0    # Axial spacing between front and rear bearings
R_PROBE_IN = 2.0             # Lever arm distance from central axis to probe center of mass (Configuration 2)
R_SHAFT_IN = 0.25            # Radius of the internal shaft the set screw bites into

# 3. Fastener / Material Parameters (Standard #4-40 Alloy Steel Set Screw)
D_SCREW_MINOR_IN = 0.0813    # Minor diameter of the set screw in inches
SY_TENSILE_PSI = 150000.0    # Tensile yield strength of alloy steel set screw (psi)
SY_SHEAR_PSI = SY_TENSILE_PSI * 0.577 # Shear yield strength (von Mises criterion)

## Static Analysis of Cantilevered OCT Probe (Configuration 1)

**Objective:** Calculate the radial reaction forces on the front ($R_1$) and rear ($R_2$) bearings due to the cantilevered weight of the OCT probe. 

**Physics First-Principles:** 
The system acts as a static beam. Because the probe's center of mass lies outside the bearing supports, it creates a pure bending moment. To maintain static equilibrium ($\Sigma M = 0$), the bearings must exert opposing radial forces, creating a reaction couple. The front bearing acts as a fulcrum (pushing up), while the rear bearing must pull down to prevent the assembly from tipping.

**Equations:**
* Front Bearing Load: $R_1 = F_{probe} \left( 1 + \frac{d}{L} \right)$
* Rear Bearing Load (Magnitude): $R_2 = F_{probe} \left( \frac{d}{L} \right)$

---

Define the System (FBD) 2D static beam problem.
- Let $F_{probe}$ be the radial force exerted by the cantilevered OCT probe.
- Let $d$ be the distance from the force $F_{probe}$ to the front bearing.
- Let $L$ be the axial spacing between the front and rear bearing (this is the variable you want to optimize).

Statics Equations:

To prevent the assembly from tilting and causing internal friction, the system must be in static equilibrium. 
The cantilevered probe creates a bending moment, which is resisted by a reaction force couple acting on the two bearings.
Take the sum of the moments about the rear bearing to find the radial load on the front bearing ($R_1$): $$\Sigma M_{rear} = 0$$
$$(R_1 \cdot L) - (F_{probe} \cdot (L + d)) = 0$$

Solving for the front bearing load: $$R_1 = F_{probe} \left( \frac{L + d}{L} \right) = F_{probe} \left( 1 + \frac{d}{L} \right)$$

Take the sum of the moments about the front bearing to find the radial load on the rear bearing ($R_2$):$$\Sigma M_{front} = 0$$

$$(R_2 \cdot L) - (F_{probe} \cdot d) = 0$$Solving for the rear bearing load:$$R_2 = F_{probe} \left( \frac{d}{L} \right)$$


In [ ]:
def calculate_bearing_reactions(F_probe, d, L):
    """
    Calculates the radial reaction forces on a two-bearing assembly.
    
    Parameters:
    F_probe (float): Downward force of the cantilevered probe (e.g., in lbf or Newtons)
    d (float): Distance from the applied force to the front bearing (e.g., in mm)
    L (float): Axial spacing between the front and rear bearings (e.g., in mm)
    """
    
    # Calculate reaction magnitudes based on static moment equilibrium
    R1 = F_probe * (1 + (d / L))
    R2 = F_probe * (d / L)
    
    print("--- Bearing Radial Reactions ---")
    print(f"Inputs: F_probe = {F_probe:.2f}, d = {d:.2f}, L = {L:.2f}")
    print(f"Front Bearing (R1) Load: {R1:.2f} (Directed UP)")
    print(f"Rear Bearing (R2) Load:  {R2:.2f} (Directed DOWN)")
    print(f"Sanity Check (Sum of Forces = 0): {R1 - R2 - F_probe:.2f}")
    
    return R1, R2	

calculate_bearing_reactions(F_probe=F_PROBE_LBF, d=D_CANTILEVER_MM, L=L_BEARING_SPACE_MM)

: 

5972K215 bearing: metric thin-section 6804 ball bearing (20 mm ID, 32 mm OD, 7 mm width).
- Dynamic Radial Load rating of 900 lbf
- Static Radial Load rating of roughly 550 lbf

Because toggling the OCT configuration is a slow, intermittent motion (not a high-speed continuous rotation like a motor shaft), your primary failure mode is the Static Load Rating ($C_0$) to prevent brinelling (permanently indenting the bearing races).

In [ ]:
def optimize_bearing_spacing(F_probe, d_range, C0_rating=550.0, FoS=2.0):
    """
    Calculates the exact minimum required bearing spacing (L) to satisfy 
    the static load constraints of the bearing, using an analytical approach.
    
    Parameters:
    F_probe (float): Estimated downward force of the probe (lbf)
    d_range (list or array): The functional constraints for the cantilever distance (mm)
    C0_rating (float): Static load rating of the selected bearing (lbf)
    FoS (float): Factor of Safety
    """
    
    # 1. Determine the maximum allowable load on the bearing
    R_allowable = C0_rating / FoS
    print(f"--- Bearing Optimization Parameters ---")
    print(f"Bearing Static Limit (C0): {C0_rating} lbf")
    print(f"Factor of Safety: {FoS}")
    print(f"Max Allowable Load (R1): {R_allowable} lbf\n")
    
    # Check if the probe itself is heavier than the allowable load
    if F_probe >= R_allowable:
        raise ValueError("Probe force exceeds allowable bearing load before any cantilevering!")
        
    print(f"{'Cantilever Dist (d) [mm]':<25} | {'Min Bearing Spacing (L) [mm]'}")
    print("-" * 55)
    
    # 2. Sweep through the required cantilever distances and calculate optimal L
    optimal_L_values = []
    
    for d in d_range:
        # Analytical solution for L
        L_min = d / ((R_allowable / F_probe) - 1)
        optimal_L_values.append(L_min)
        
        print(f"{d:<25.2f} | {L_min:.4f}")
        
    return optimal_L_values

# Example Execution:
# Assume the probe exerts a 10 lbf load. 
# We evaluate functional cantilever distances (d) from 20mm to 50mm in 10mm increments.
d_constraints = np.array([20.0, 30.0, 40.0, 50.0])
optimize_bearing_spacing(F_probe=F_PROBE_LBF, d_range=d_constraints)

## Set Screw Shear Stress Analysis (Configuration 2)

**Objective:** Calculate the shear stress applied to the locking set screw when the OCT probe is toggled 90 degrees orthogonally.

**Physics First-Principles:** 
When the probe is rotated 90 degrees, gravity still pulls down, converting the cantilever load into a combined bending and torsional load. Standard radial bearings cannot resist this torque. Static equilibrium ($\Sigma \tau = 0$) is maintained entirely by the set screw. The applied torque from the probe is countered by a shear force acting on the cross-sectional area of the set screw tip.

**Equations:**
* Force on Screw: $F_{screw} = F_{probe} \left( \frac{r_{probe}}{r_{screw}} \right)$
* Cross-sectional Area: $A_{screw} = \frac{\pi \cdot d_{screw}^2}{4}$
* Shear Stress: $\tau_{shear} = \frac{F_{screw}}{A_{screw}}$

---

**Define the Variables**
- $F_{probe}$: The weight of the probe (Force = mass $\cdot$ gravity).
- $r_{probe}$: The horizontal distance (lever arm) from the central axis of rotation to the probe's center of mass.
- $r_{screw}$: The distance from the central axis of rotation to the shear plane of the set screw (essentially the radius of the internal shaft the screw is biting into).
- $d_{screw}$: The minor diameter of the set screw.

**Equations** 
- Applied Torque: gravity pulls down on the probe at a distance of $r_{probe}$ from the axis, applied torque ($\tau_{applied}$) twisting the assembly: $$\tau_{applied} = F_{probe} \cdot r_{probe}$$
- Reaction Force on the Set Screw To maintain static equilibrium ($\Sigma \tau = 0$), the set screw must provide an equal and opposite reaction torque: $$\tau_{screw} = F_{screw} \cdot r_{screw}$$
- Setting them equal to each other: $$F_{screw} \cdot r_{screw} = F_{probe} \cdot r_{probe}$$
- Solving for the physical force acting on the tip of the screw:$$F_{screw} = F_{probe} \left( \frac{r_{probe}}{r_{screw}} \right)$$

Because the probe is sticking out much further than the radius of the internal shaft, this ratio acts as a mechanical disadvantage. The force shearing the set screw ($F_{screw}$) will be a massive multiplier of the probe's actual weight.


**Failure Modes**

check static yield limit to ensure the screw doesn't permanently deform or shear.
- Shear Stress on the Screw:The rotational force is trying to cut the set screw in half (like a pair of scissors). You calculate the shear stress ($\tau_{shear}$) by dividing the force by the cross-sectional area of the screw ($A_{screw}$):$$\tau_{shear} = \frac{F_{screw}}{A_{screw}} = \frac{F_{screw}}{\frac{\pi \cdot d_{screw}^2}{4}}$$
- Bearing Stress on the Shaft:The tip of the set screw is pressing into a small patch of the internal shaft. If the force $F_{screw}$ is too high, the screw will gouge and permanently deform the shaft (this is called bearing stress or compressive yield).If you explain to the interviewer that you chose a hardened steel set screw and verified that the shear stress ($\tau_{shear}$) was well below the material's yield strength, you will demonstrate a rock-solid grasp of mechanics of materials.

In [ ]:
def calculate_set_screw_shear(F_probe, r_probe, r_screw, d_screw):
    """
    Calculates the shear stress acting on a set screw resisting rotation.
    
    Parameters:
    F_probe (float): Downward force of the probe (e.g., in lbf)
    r_probe (float): Moment arm distance from the central axis to the probe's center of mass (e.g., in inches)
    r_screw (float): Radius of the internal shaft the screw bites into (e.g., in inches)
    d_screw (float): Minor diameter of the set screw (e.g., in inches)
    """
    
    # 1. Calculate the mechanical disadvantage (Force multiplier)
    F_screw = F_probe * (r_probe / r_screw)
    
    # 2. Calculate the cross-sectional area of the set screw (shear plane)
    A_screw = (math.pi * (d_screw ** 2)) / 4
    
    # 3. Calculate shear stress (Force / Area)
    tau_shear = F_screw / A_screw
    
    print("--- Set Screw Shear Stress ---")
    print(f"Applied Torque: {F_probe * r_probe:.2f} in-lbf")
    print(f"Reaction Force on Screw Tip (F_screw): {F_screw:.2f} lbf")
    print(f"Screw Cross-Sectional Area: {A_screw:.5f} sq in")
    print(f"Shear Stress (tau_shear): {tau_shear:.2f} psi")
    
    return tau_shear

calculate_set_screw_shear(F_probe=F_PROBE_LBF, r_probe=R_PROBE_IN, r_screw=R_SHAFT_IN, d_screw=D_SCREW_MINOR_IN)

In [ ]:
# Material: 316 Stainless Steel
SY_TENSILE_PSI = 30000.0 
SY_SHEAR_PSI = SY_TENSILE_PSI * 0.577 
TARGET_FOS = 3.0

# Database of standard machine screw minor diameters (inches)
STANDARD_SCREWS = {
    "#0-80": 0.0465,
    "#2-56": 0.0641,
    "#4-40": 0.0813,
    "#6-32": 0.0997,
    "#8-32": 0.1257,
    "#10-32": 0.1593,
    "1/4-20": 0.1850
}

def spec_optimal_fastener(F_probe, r_probe, r_shaft, Sy_shear, target_FoS, screw_db):
    """
    Analytically determines the minimum required screw diameter based on material 
    limits and target FoS, then selects the optimal standard off-the-shelf fastener.
    """
    print("--- Torsional Locking Mechanism Optimization ---")
    
    # 1. Calculate applied mechanical loads
    F_screw = F_probe * (r_probe / r_shaft)
    print(f"Reaction Force on Screw Tip: {F_screw:.2f} lbf")
    
    # 2. Determine allowable stress and required geometry
    tau_allowable = Sy_shear / target_FoS
    A_req = F_screw / tau_allowable
    d_req = math.sqrt((4 * A_req) / math.pi)
    
    print(f"Target Factor of Safety:     {target_FoS:.1f}")
    print(f"Allowable Shear Stress:      {tau_allowable:,.2f} psi")
    print(f"Absolute Min Minor Diameter: {d_req:.5f} in\n")
    
    # 3. Select the optimal standard screw
    selected_screw = None
    selected_dia = None
    
    for name, minor_dia in screw_db.items():
        if minor_dia >= d_req:
            selected_screw = name
            selected_dia = minor_dia
            break # Stop at the first (smallest) screw that works
            
    if selected_screw:
        actual_area = (math.pi * (selected_dia ** 2)) / 4
        actual_stress = F_screw / actual_area
        actual_FoS = Sy_shear / actual_stress
        
        print("--- Optimal Fastener Specification ---")
        print(f"Selected Fastener:           {selected_screw} (316 Stainless Steel)")
        print(f"Minor Diameter:              {selected_dia:.4f} in")
        print(f"Actual Shear Stress:         {actual_stress:,.2f} psi")
        print(f"Actual Factor of Safety:     {actual_FoS:.2f} (PASS)")
    else:
        print("STATUS: FAIL | Load exceeds all standard screws in the database.")
        print("Redesign required: Increase shaft radius or use multiple locking pins.")

# Execute the optimization
spec_optimal_fastener(
    F_probe=F_PROBE_LBF, 
    r_probe=R_PROBE_IN, 
    r_shaft=R_SHAFT_IN, 
    Sy_shear=SY_SHEAR_PSI, 
    target_FoS=TARGET_FOS, 
    screw_db=STANDARD_SCREWS
)

: 